# KernelSim: OS scheduling and paging simulator
Run the cells top to bottom. Change the inputs and re-run to experiment.

## 1. Core simulator

In [ ]:
"""KernelSim core: CPU scheduling + page replacement simulators."""
from dataclasses import dataclass
from typing import List, Tuple, Dict


@dataclass
class Proc:
    pid: str
    arrival: int
    burst: int
    priority: int = 1


def _metrics(procs, finish, first, gantt):
    rows = []
    for p in procs:
        tat = finish[p.pid] - p.arrival
        rows.append({"PID": p.pid, "Arrival": p.arrival, "Burst": p.burst,
                     "Priority": p.priority, "Completion": finish[p.pid],
                     "Turnaround": tat, "Waiting": tat - p.burst,
                     "Response": first[p.pid] - p.arrival})
    end = max(finish.values())
    busy = sum(e - s for pid, s, e in gantt if pid != "idle")
    n = len(rows)
    summary = {"Avg waiting": sum(r["Waiting"] for r in rows) / n,
               "Avg turnaround": sum(r["Turnaround"] for r in rows) / n,
               "Avg response": sum(r["Response"] for r in rows) / n,
               "Throughput": n / end,
               "CPU utilisation %": 100 * busy / end}
    return rows, summary


def _push(g, pid, s, e):
    if g and g[-1][0] == pid and g[-1][2] == s:
        g[-1] = (pid, g[-1][1], e)
    else:
        g.append((pid, s, e))


def schedule(procs: List[Proc], algo: str, quantum: int = 2,
             mlfq_quanta=(2, 4, 8), preemptive: bool = False):
    """Returns (gantt[(pid,start,end)], rows, summary). algo in
    FCFS, SJF, Priority, RR, MLFQ. preemptive applies to SJF (SRTF) / Priority."""
    rem = {p.pid: p.burst for p in procs}
    first, finish, g = {}, {}, []
    t, done = 0, 0
    n = len(procs)
    by = {p.pid: p for p in procs}

    if algo in ("RR", "MLFQ"):
        levels = len(mlfq_quanta) if algo == "MLFQ" else 1
        q = [[] for _ in range(levels)]
        lvl = {p.pid: 0 for p in procs}
        arrived = set()

        def admit(upto):
            for p in sorted(procs, key=lambda x: (x.arrival, x.pid)):
                if p.arrival <= upto and p.pid not in arrived:
                    arrived.add(p.pid); q[0].append(p.pid)
        admit(t)
        while done < n:
            cur = next((i for i in range(levels) if q[i]), None)
            if cur is None:
                nxt = min(p.arrival for p in procs if p.pid not in arrived)
                _push(g, "idle", t, nxt); t = nxt; admit(t); continue
            pid = q[cur].pop(0)
            slice_ = (mlfq_quanta[cur] if algo == "MLFQ" else quantum)
            run = min(slice_, rem[pid])
            first.setdefault(pid, t)
            _push(g, pid, t, t + run); t += run; rem[pid] -= run
            admit(t)
            if rem[pid] == 0:
                finish[pid] = t; done += 1
            elif algo == "MLFQ":
                nl = min(cur + 1, levels - 1); lvl[pid] = nl; q[nl].append(pid)
            else:
                q[0].append(pid)
    else:
        key = {"FCFS": lambda p: (p.arrival, p.pid),
               "SJF": lambda p: (rem[p.pid], p.arrival),
               "Priority": lambda p: (p.priority, p.arrival)}[algo]
        pre = preemptive and algo != "FCFS"
        while done < n:
            ready = [p for p in procs if p.arrival <= t and rem[p.pid] > 0]
            if not ready:
                nxt = min(p.arrival for p in procs if rem[p.pid] > 0)
                _push(g, "idle", t, nxt); t = nxt; continue
            p = min(ready, key=key)
            first.setdefault(p.pid, t)
            if pre:
                fut = [x.arrival for x in procs if x.arrival > t]
                run = min(rem[p.pid], (min(fut) - t) if fut else rem[p.pid])
            else:
                run = rem[p.pid]
            _push(g, p.pid, t, t + run); t += run; rem[p.pid] -= run
            if rem[p.pid] == 0:
                finish[p.pid] = t; done += 1
    rows, summary = _metrics(procs, finish, first, g)
    return g, rows, summary


def page_replace(refs: List[int], frames: int, algo: str):
    """algo in FIFO, LRU, OPT. Returns (steps, faults, fault_rate, hit_rate).
    steps: list of dicts {ref, frames, fault, evicted}."""
    mem: List[int] = []
    last: Dict[int, int] = {}
    steps, faults = [], 0
    for i, r in enumerate(refs):
        fault, ev = r not in mem, None
        if fault:
            faults += 1
            if len(mem) == frames:
                if algo == "FIFO":
                    ev = mem.pop(0)
                elif algo == "LRU":
                    ev = min(mem, key=lambda x: last[x]); mem.remove(ev)
                else:  # OPT
                    def nxt(x):
                        try: return refs.index(x, i + 1)
                        except ValueError: return float("inf")
                    ev = max(mem, key=nxt); mem.remove(ev)
            mem.append(r)
        last[r] = i
        steps.append({"ref": r, "frames": list(mem), "fault": fault, "evicted": ev})
    n = len(refs)
    return steps, faults, faults / n, 1 - faults / n


# ---------------------------------------------------------------- deadlock
import random

BANKER_ALLOC = [[0, 1, 0], [2, 0, 0], [3, 0, 2], [2, 1, 1], [0, 0, 2]]
BANKER_MAX = [[7, 5, 3], [3, 2, 2], [9, 0, 2], [2, 2, 2], [4, 3, 3]]


def bankers(available, alloc=BANKER_ALLOC, mx=BANKER_MAX):
    """Banker's safety algorithm. Returns (safe, order, trace, need).
    trace rows: (process, free_before, free_after)."""
    need = [[m - a for m, a in zip(mr, ar)] for mr, ar in zip(mx, alloc)]
    work, done, trace = list(available), [], []
    while len(done) < len(alloc):
        i = next((k for k in range(len(alloc)) if k not in done
                  and all(n <= w for n, w in zip(need[k], work))), None)
        if i is None:
            break
        before = list(work)
        work = [w + a for w, a in zip(work, alloc[i])]
        trace.append((f"P{i}", before, list(work)))
        done.append(i)
    return len(done) == len(alloc), [f"P{i}" for i in done], trace, need


# ------------------------------------------------------- memory performance
def effective_access_time(hit, tlb_ns=10, mem_ns=100):
    """Single-level page table: a TLB miss costs one extra memory access."""
    return hit * (tlb_ns + mem_ns) + (1 - hit) * (tlb_ns + 2 * mem_ns)


def cpu_utilisation(io_wait, n):
    """Classic model: utilisation = 1 - p**n for n processes each waiting on I/O a fraction p."""
    return 1 - io_wait ** n


# ------------------------------------------------------------- concurrency
def race(lock, n=1000, threads=2, rng=random):
    """Final value of a shared counter when each thread increments it n times.
    Each increment is load, add, store; without a lock the steps interleave."""
    c, reg, pc = 0, [0] * threads, [0] * threads
    left, holder = [n] * threads, -1
    while any(left):
        act = [i for i in range(threads) if left[i] and (not lock or holder < 0 or holder == i)]
        i = rng.choice(act)
        if lock and holder < 0 and pc[i] == 0:
            holder = i
        if pc[i] == 0:
            reg[i] = c
        elif pc[i] == 1:
            reg[i] += 1
        else:
            c = reg[i]; left[i] -= 1
            if lock:
                holder = -1
        pc[i] = (pc[i] + 1) % 3
    return c


def dining(strategy="naive", steps=300, n=5, seed=None):
    """Dining philosophers under random scheduling. strategy: naive | ordered.
    Returns (meals_over_time, deadlock_step_or_None, final_states) where
    states are 0 thinking, 1 holding first fork, 2 eating."""
    rng = random.Random(seed)
    owner, state, meals, hist, dead = [None] * n, [0] * n, 0, [], None
    for t in range(steps):
        if dead is None and all(s == 1 for s in state):
            dead = t
        i = rng.randrange(n)
        l, r = i, (i + 1) % n
        a, b = (l, r) if strategy == "naive" else (min(l, r), max(l, r))
        if state[i] == 0 and owner[a] is None:
            owner[a] = i; state[i] = 1
        elif state[i] == 1 and owner[b] is None:
            owner[b] = i; state[i] = 2
        elif state[i] == 2:
            owner[a] = owner[b] = None; state[i] = 0; meals += 1
        hist.append(meals)
    return hist, dead, state


# ------------------------------------------------------ memory allocation
def allocate(blocks, requests, strategy="first"):
    """Contiguous allocation into free blocks. strategy: first | best | worst.
    Returns (remaining_per_block, placed_per_block[(request_idx, size)], failed_request_idxs)."""
    rem = list(blocks)
    placed = [[] for _ in blocks]
    failed = []
    for i, q in enumerate(requests):
        cand = [(r, k) for k, r in enumerate(rem) if r >= q]
        if not cand:
            failed.append(i)
            continue
        if strategy == "first":
            r, k = cand[0]
        elif strategy == "best":
            r, k = min(cand)
        else:
            r, k = max(cand, key=lambda x: (x[0], -x[1]))
        rem[k] -= q
        placed[k].append((i, q))
    return rem, placed, failed


def page_size_overhead(proc_bytes, entry_bytes, page_bytes):
    """Returns (page_table_bytes, internal_fragmentation_bytes, total) for one process."""
    table = proc_bytes * entry_bytes / page_bytes
    frag = page_bytes / 2
    return table, frag, table + frag


def optimal_page_size(proc_bytes, entry_bytes):
    return (2 * proc_bytes * entry_bytes) ** 0.5


# -------------------------------------------------------------- benchmarks
def gen_workload(rng):
    """Random workload of 4-5 processes with no idle gaps (arrival <= work queued so far)."""
    n = rng.randint(4, 5)
    S, a, out = 0, 0, []
    for i in range(n):
        b = rng.randint(2, 9)
        a = 0 if i == 0 else a + rng.randint(0, S - a)
        out.append(Proc(f"P{i + 1}", a, b, rng.randint(1, 4)))
        S += b
    return out


def monte_carlo(n=300, seed=None, algos=("FCFS", "SJF", "Priority", "RR", "MLFQ")):
    """Run every scheduler on n random workloads.
    Returns (results[algo] = {'wait': [...], 'resp': [...]}, wins[algo]) where a win is the lowest
    average waiting time (ties count for every tied algorithm)."""
    rng = random.Random(seed)
    res = {a: {"wait": [], "resp": []} for a in algos}
    wins = dict.fromkeys(algos, 0)
    for _ in range(n):
        w = gen_workload(rng)
        ws = {}
        for a in algos:
            _, _, s = schedule(w, a)
            res[a]["wait"].append(s["Avg waiting"])
            res[a]["resp"].append(s["Avg response"])
            ws[a] = s["Avg waiting"]
        m = min(ws.values())
        for a in algos:
            wins[a] += ws[a] == m
    return res, wins


# --------------------------------------------------------- disk scheduling
def disk_schedule(requests, head, algo="FCFS", disk_size=200):
    """algo: FCFS | SSTF | SCAN (sweeps down first) | CSCAN (sweeps up, wraps to 0).
    Returns (path, total_head_movement)."""
    reqs = list(requests)
    upper = sorted(r for r in reqs if r > head)
    lower = sorted((r for r in reqs if r <= head), reverse=True)
    if algo == "FCFS":
        order = reqs
    elif algo == "SSTF":
        left, pos, order = reqs[:], head, []
        while left:
            nxt = min(left, key=lambda r: abs(r - pos))
            left.remove(nxt); order.append(nxt); pos = nxt
    elif algo == "SCAN":
        order = lower + ([0] if lower and upper else []) + upper
    else:
        order = upper + ([disk_size - 1, 0] + lower[::-1] if lower else [])
    path = [head] + order
    return path, sum(abs(a - b) for a, b in zip(path, path[1:]))


# ------------------------------------------------- parallel speedup, tables
def amdahl(parallel_fraction, cores):
    return 1 / ((1 - parallel_fraction) + parallel_fraction / cores)


def page_table_kb(used_mb, va_bits=32, page_kb=4, entry_bytes=4):
    """Memory for a flat page table vs a two-level one when used_mb of address space is mapped
    contiguously. Returns (flat_kb, two_level_kb)."""
    entries = page_kb * 1024 // entry_bytes
    flat = 2 ** va_bits // (page_kb * 1024) * entry_bytes / 1024
    l2 = min(entries, -(-used_mb * 1024 // (page_kb * entries)))
    return flat, page_kb + l2 * page_kb


# ---------------------------------------------------------------- live runner
def simulate_ticks(procs, algo="RR", quantum=2):
    """Tick-by-tick CPU simulation for the live runners.
    algo: FCFS | SJF | SRTF | PRIO | PRIO-P | RR | MLFQ  (SRTF and PRIO-P are preemptive).
    Returns {'frames': [...], 'first': [...], 'fin': [...], 'end': int}. frames[t] is the state at
    the START of tick t: {'t', 'run' (index or -1), 'ready' (indices, next first), 'rem', 'level'};
    the last frame is the final state after every process has finished."""
    n = len(procs)
    rem = [p.burst for p in procs]
    first, fin = [None] * n, [None] * n
    qs, q, ml, lvl = (2, 4, 8), [], [[], [], []], [0] * n
    cur, slice_, pend, t, done, frames = -1, 0, -1, 0, 0, []
    pre = algo in ("SRTF", "PRIO-P")
    by_rem = lambda i: (rem[i], procs[i].arrival)
    by_pri = lambda i: (procs[i].priority, procs[i].arrival)
    key = {"FCFS": lambda i: (procs[i].arrival,), "SJF": by_rem, "SRTF": by_rem,
           "PRIO": by_pri, "PRIO-P": by_pri}.get(algo)
    while done < n and t < 10000:
        for i, p in enumerate(procs):
            if p.arrival == t:
                (q if algo == "RR" else ml[0]).append(i) if algo in ("RR", "MLFQ") else None
        if pend >= 0:
            (q if algo == "RR" else ml[lvl[pend]]).append(pend)
            pend = -1
        ready = [i for i in range(n) if procs[i].arrival <= t and rem[i] > 0]
        run = -1
        if algo == "RR":
            if cur >= 0: run = cur
            elif q: run, slice_ = q.pop(0), 0
        elif algo == "MLFQ":
            if cur >= 0: run = cur
            else:
                l = next((k for k in range(3) if ml[k]), None)
                if l is not None: run, slice_ = ml[l].pop(0), 0
        elif cur >= 0 and not pre:
            run = cur
        elif ready:
            run = min(ready, key=key)
            if cur >= 0 and pre and cur in ready and key(cur) == key(run): run = cur
        if algo == "RR": waiting = list(q)
        elif algo == "MLFQ": waiting = [i for lv in ml for i in lv]
        else: waiting = sorted((i for i in ready if i != run), key=key)
        frames.append({"t": t, "run": run, "ready": waiting, "rem": list(rem),
                       "level": lvl[run] if run >= 0 else 0})
        cur = run
        if run >= 0:
            if first[run] is None: first[run] = t
            rem[run] -= 1; slice_ += 1
            if rem[run] == 0:
                fin[run] = t + 1; done += 1; cur = -1
            elif algo == "RR" and slice_ >= quantum:
                pend, cur = run, -1
            elif algo == "MLFQ" and slice_ >= qs[lvl[run]]:
                lvl[run] = min(lvl[run] + 1, 2); pend, cur = run, -1
        t += 1
    frames.append({"t": t, "run": -1, "ready": [], "rem": list(rem), "level": 0})
    return {"frames": frames, "first": first, "fin": fin, "end": t}

print('KernelSim loaded')

## 2. CPU scheduling
Edit `procs` and compare algorithms by waiting, turnaround, response time, throughput and CPU utilisation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

procs = [Proc('P1',0,5,2), Proc('P2',1,3,1), Proc('P3',2,8,3), Proc('P4',3,6,2)]
QUANTUM = 2

def gantt(g, procs, title):
    ids = [p.pid for p in procs]; cols = plt.cm.Set3.colors
    fig, ax = plt.subplots(figsize=(10, 1.2))
    for pid, s, e in g:
        c = '#dddddd' if pid == 'idle' else cols[ids.index(pid) % 12]
        ax.barh(0, e - s, left=s, color=c, edgecolor='k')
        ax.text((s + e) / 2, 0, pid, ha='center', va='center', fontsize=8)
    ax.set_yticks([]); ax.set_xlabel('time'); ax.set_title(title); plt.show()

results = {}
for a in ['FCFS', 'SJF', 'Priority', 'RR', 'MLFQ']:
    g, rows, s = schedule(procs, a, quantum=QUANTUM)
    gantt(g, procs, a)
    results[a] = s
cmp = pd.DataFrame(results).T.round(2)
cmp

In [ ]:
cmp[['Avg waiting','Avg turnaround','Avg response']].plot.bar(figsize=(8,4), rot=0, title='Scheduling comparison');

## 3. Page replacement
FIFO vs LRU vs Optimal on a reference string.

In [ ]:
refs = [7,0,1,2,0,3,0,4,2,3,0,3,2,1,2,0,1,7,0,1]
FRAMES = 3
for a in ['FIFO', 'LRU', 'OPT']:
    steps, f, fr, hr = page_replace(refs, FRAMES, a)
    print(f'{a}: {f} faults, fault rate {fr:.0%}, hit rate {hr:.0%}')
steps, *_ = page_replace(refs, FRAMES, 'LRU')
pd.DataFrame([{'ref': s['ref'], 'frames': s['frames'], 'result': 'fault' if s['fault'] else 'hit', 'evicted': s['evicted']} for s in steps])

## 4. Faults vs. frame count
Try the classic Belady string `1 2 3 4 1 2 5 1 2 3 4 5`: FIFO gets worse going from 3 to 4 frames.

In [ ]:
belady = [1,2,3,4,1,2,5,1,2,3,4,5]
sweep = pd.DataFrame({a: [page_replace(belady, k, a)[1] for k in range(1, 8)] for a in ['FIFO','LRU','OPT']}, index=range(1, 8))
sweep.index.name = 'frames'
sweep.plot(marker='o', figsize=(8,4), title="Page faults vs frames (Belady's anomaly)");
sweep

## 5. Deadlock avoidance (Banker's algorithm)
Try other `available` vectors.

In [ ]:
for available in ([3,3,2], [2,2,1], [1,1,1], [0,0,0]):
    safe, order, trace, need = bankers(available)
    print(available, '-> safe, order:' if safe else '-> UNSAFE, finished only:', ' > '.join(order))
safe, order, trace, need = bankers([3,3,2])
pd.DataFrame(trace, columns=['runs to completion', 'free before', 'free after'])

## 6. Memory performance
Effective access time against TLB hit ratio, and CPU utilisation against the number of processes in memory.

In [ ]:
xs = list(range(50, 101, 5))
plt.figure(figsize=(8,3.5)); plt.plot(xs, [effective_access_time(x/100) for x in xs], marker='o')
plt.xlabel('TLB hit ratio (%)'); plt.ylabel('effective access time (ns)'); plt.grid(alpha=.3); plt.show()

ns = range(1, 11)
plt.figure(figsize=(8,3.5))
for p in (0.2, 0.5, 0.8):
    plt.plot(ns, [100*cpu_utilisation(p, n) for n in ns], marker='o', label=f'{int(p*100)}% I/O wait')
plt.xlabel('processes in memory'); plt.ylabel('CPU utilisation (%)'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 7. Concurrency
Two threads add 1 to a shared counter 1,000 times. The correct total is 2,000.

In [ ]:
trials = pd.DataFrame({'no lock': [race(False) for _ in range(10)], 'mutex': [race(True) for _ in range(10)]}, index=range(1, 11))
trials.plot.bar(figsize=(8,3.5), rot=0, title='Final counter value'); plt.axhline(2000, color='k', ls='--'); plt.show()
print('average without lock:', trials['no lock'].mean())

In [ ]:
for strat in ('naive', 'ordered'):
    runs = [dining(strat, 300) for _ in range(200)]
    dead = [d for _, d, _ in runs if d is not None]
    print(f'{strat}: deadlocked in {len(dead)}/200 runs')
    plt.plot(runs[0][0], label=strat)
plt.xlabel('time step'); plt.ylabel('meals eaten'); plt.legend(); plt.title('Dining philosophers (one run each)'); plt.show()

## 8. Memory allocation: first, best and worst fit
Free blocks of 100, 500, 200, 300 and 600 KB; requests of 212, 417, 112 and 426 KB.

In [ ]:
blocks, reqs = [100, 500, 200, 300, 600], [212, 417, 112, 426]
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
rows = []
for ax, strat in zip(axes, ['first', 'best', 'worst']):
    rem, placed, failed = allocate(blocks, reqs, strat)
    for k, b in enumerate(blocks):
        left = 0
        for i, q in placed[k]:
            ax.barh(k, q, left=left, color=plt.cm.Set2.colors[i], edgecolor='k')
            ax.text(left + q/2, k, f'R{i+1}', ha='center', va='center', fontsize=8); left += q
        ax.barh(k, b - left, left=left, color='none', edgecolor='gray', linestyle='--')
    ax.set_title(f'{strat}-fit', loc='left', fontsize=10)
    ax.set_yticks(range(len(blocks))); ax.set_yticklabels([f'{b} KB' for b in blocks]); ax.invert_yaxis()
    rows.append({'strategy': strat, 'failed requests': [f'R{i+1}' for i in failed], 'free left (KB)': sum(rem), 'largest free chunk (KB)': max(rem)})
axes[-1].set_xlabel('KB'); plt.tight_layout(); plt.show()
pd.DataFrame(rows)

## 9. Choosing a page size
Small pages waste little memory inside a page but need a big page table. Large pages do the opposite.

In [ ]:
proc, entry = 1024*1024, 8   # 1 MB process, 8-byte page-table entries
sizes = [2**k for k in range(8, 17)]
ov = pd.DataFrame([page_size_overhead(proc, entry, p) for p in sizes], columns=['page table', 'internal fragmentation', 'total'], index=sizes)
ov.plot(logx=True, marker='o', figsize=(8,4), title='Overhead per process (bytes) vs page size'); plt.xlabel('page size (bytes)'); plt.grid(alpha=.3); plt.show()
print('best page size is about', round(optimal_page_size(proc, entry)), 'bytes')

## 10. Scheduler benchmark on random workloads
Every algorithm runs on the same random workloads, so the comparison is fair.

In [ ]:
import numpy as np
res, wins = monte_carlo(500, seed=1)
summary = pd.DataFrame({a: {'mean waiting': np.mean(r['wait']), 'median waiting': np.median(r['wait']),
                            '90th pct waiting': np.percentile(r['wait'], 90), 'mean response': np.mean(r['resp']),
                            '% workloads with lowest waiting': 100*wins[a]/500} for a, r in res.items()}).T.round(2)
display(summary)
plt.figure(figsize=(8,4)); plt.boxplot([r['wait'] for r in res.values()], labels=list(res)); plt.ylabel('average waiting time'); plt.title('Waiting time across 500 random workloads'); plt.show()

## 11. Disk scheduling
Head starts at cylinder 53 on a 200-cylinder disk.

In [ ]:
reqs, head = [98, 183, 37, 122, 14, 124, 65, 67], 53
plt.figure(figsize=(9,3.5))
for a, label in [('FCFS','FCFS'), ('SSTF','SSTF'), ('SCAN','SCAN'), ('CSCAN','C-SCAN')]:
    path, total = disk_schedule(reqs, head, a)
    plt.plot(path, marker='o', label=f'{label} ({total})')
plt.xlabel('step'); plt.ylabel('cylinder'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 12. Amdahl's law and page-table memory

In [ ]:
cores = [1, 2, 4, 8, 16, 32, 64]
plt.figure(figsize=(8,3.5))
for p in (0.5, 0.9, 0.95, 0.99):
    plt.plot(cores, [amdahl(p, c) for c in cores], marker='o', label=f'{int(p*100)}% parallel')
plt.xscale('log', base=2); plt.xlabel('cores'); plt.ylabel('speedup'); plt.legend(); plt.grid(alpha=.3); plt.show()

used = [4, 16, 64, 256, 1024, 4096]
tabs = [page_table_kb(u) for u in used]
plt.figure(figsize=(8,3.5))
plt.plot(used, [t[0] for t in tabs], marker='o', label='flat'); plt.plot(used, [t[1] for t in tabs], marker='o', label='two-level')
plt.xscale('log', base=2); plt.xlabel('mapped memory (MB)'); plt.ylabel('page-table memory (KB)'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 13. Live runner
Press the play button or drag the slider to watch the CPU tick by tick. Change `procs` and `ALGO` and run the cell again.

`ALGO` can be `FCFS`, `SJF`, `SRTF`, `PRIO`, `PRIO-P`, `RR` or `MLFQ`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

procs = [Proc('P1',0,5,2), Proc('P2',1,3,1), Proc('P3',2,8,3), Proc('P4',3,6,2)]
ALGO, QUANTUM = 'RR', 2

sim = simulate_ticks(procs, ALGO, QUANTUM)
frames, end = sim['frames'], sim['end']
cols = plt.cm.Set3.colors
out = widgets.Output()

def draw(t):
    f = frames[t]
    with out:
        clear_output(wait=True)
        cpu = procs[f['run']].pid if f['run'] >= 0 else ('idle' if t < end else 'all done')
        print(f"t = {t}    CPU: {cpu}    ready queue: {' > '.join(procs[i].pid for i in f['ready']) or 'empty'}")
        fig, ax = plt.subplots(figsize=(10, 1.4))
        for k in range(t):
            r = frames[k]['run']
            if r >= 0:
                ax.barh(0, 1, left=k, color=cols[r % 12], edgecolor='k'); ax.text(k+.5, 0, procs[r].pid, ha='center', va='center', fontsize=7)
        ax.axvline(t, color='r'); ax.set_xlim(0, end); ax.set_yticks([]); ax.set_xlabel('time'); plt.show()
        rows = []
        for i, p in enumerate(procs):
            state = 'done' if f['rem'][i] == 0 else 'running' if f['run'] == i else 'waiting' if p.arrival <= t else 'not arrived'
            rows.append({'process': p.pid, 'state': state, 'remaining': f['rem'][i], 'burst': p.burst})
        display(pd.DataFrame(rows))

play = widgets.Play(min=0, max=end, step=1, interval=600)
slider = widgets.IntSlider(min=0, max=end, description='time', layout=widgets.Layout(width='60%'))
widgets.jslink((play, 'value'), (slider, 'value'))
slider.observe(lambda c: draw(c['new']), names='value')
display(widgets.HBox([play, slider]), out)
draw(0)